# 01 — Análise exploratória

POLIDriving: níveis 1/2 → baixo risco; 3/4 → alto risco. A inspeção estrutural global não orienta seleção de features. As análises abaixo usam **apenas o treino** do split fixado em seed 42. Execute `python -m src.data_loader` antes deste notebook.

In [ ]:
from pathlib import Path
import sys
ROOT = Path.cwd()
if not (ROOT / 'src').is_dir():
    ROOT = ROOT.parent
if not (ROOT / 'src/config.py').is_file():
    raise RuntimeError('Abra o notebook a partir da raiz ou da pasta notebooks.')
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import display, Image, Markdown
from src.config import ExperimentConfig, TARGET
config = ExperimentConfig(output=ROOT)


In [ ]:
from src.train import prepare_data
from src.data_loader import inspect_data
from src.eda import run_eda
frame, split = prepare_data(config)
train = frame.loc[split['train_indices']]
print('Treino:', train.shape, 'Teste reservado:', len(split['test_indices']))
display(train.head())
display(train.tail())
train.info()
display(train.describe(include='all'))

## Ausências, duplicatas e classes

Distinguir o balanceamento publicado da prevalência real. `dataset_inspection.json` registra duplicatas antes da remoção; a tabela abaixo descreve o treino já separado.

In [ ]:
inspection = inspect_data(train)
display(pd.DataFrame({'missing_count': inspection['missing_count'], 'missing_percentage': inspection['missing_percentage']}))
print('Duplicatas no treino:', inspection['duplicates'])
print('Classes originais:', inspection['original_classes'])
print('Classes binárias:', inspection['binary_classes'])
print('Percentuais:', inspection['binary_percentages'])
run_eda(train, ROOT)

In [ ]:
for name in ['01_class_distribution', '02_missing_values', '04_feature_distributions']:
    display(Image(filename=str(ROOT / f'reports/figures/{name}.png')))

## Relações e outliers

Boxplots contrastam as classes; IQR identifica extremos sem removê-los. A assimetria é calculada por atributo. Um extremo de aceleração pode ser o próprio sinal de risco procurado. Correlação não implica causalidade; atributos constantes não têm correlação definida.

In [ ]:
display(Image(filename=str(ROOT / 'reports/figures/12_features_by_class.png')))
display(pd.read_csv(ROOT / 'reports/metrics/outliers.csv'))
display(Image(filename=str(ROOT / 'reports/figures/03_correlation_matrix.png')))

In [ ]:
corr = train.select_dtypes('number').drop(columns=TARGET).corr()
pairs = [(a, b, corr.loc[a,b]) for i,a in enumerate(corr) for b in corr.columns[i+1:] if abs(corr.loc[a,b]) >= .8]
display(pd.DataFrame(pairs, columns=['feature_a','feature_b','correlation']))
display(pd.read_csv(ROOT / 'reports/metrics/features_by_class.csv', header=[0,1], index_col=0))

## Questões para discussão

Quais atributos têm dispersão diferente entre classes? Quais correlações indicam redundância? Há extremos compatíveis com situações perigosas? Confronte esses achados com a origem sintética/parcialmente baseada em regras dos rótulos. Nenhuma conclusão desta EDA modifica o conjunto de teste ou remove features automaticamente.